# Jewelry Ridge Regression Pricer

Train and evaluate jewelry Ridge regression baselines with and without auction estimate features.

## Configuration

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "jewelry_ridge_pricer").exists():
    PROJECT_DIR = NOTEBOOK_DIR
else:
    PROJECT_DIR = Path("Linear_Regression/jewelry/Ridge_Regression").resolve()

sys.path.insert(0, str(PROJECT_DIR))

OUTPUT_DIR = PROJECT_DIR / "outputs"
WORKSPACE_ROOT = PROJECT_DIR.parents[2]
JEWELRY_DATA_DIR = WORKSPACE_ROOT / "Data" / "sampled_sets_jewelry"
TRAIN_FILE = JEWELRY_DATA_DIR / "Jewelry_2025_high_midpoint_error_remainder_training.csv"
QUARTERS = ("2025Q1", "2025Q2", "2025Q3", "2025Q4")
TEST_FILES = {
    quarter: JEWELRY_DATA_DIR / f"Jewelry_2025_high_midpoint_error_q{quarter}_sample1000.csv"
    for quarter in QUARTERS
}
TARGET_COL = "ah_price_usd"
TRAIN_BEFORE = None
TRAIN_ROWS = None
TEST_ROWS_PER_QUARTER = None
MIN_CATEGORY_COUNT = 20
VALIDATION_FRACTION = 0.10
RANDOM_STATE = 42
VERBOSE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Train file:  {TRAIN_FILE}")
for quarter, path in TEST_FILES.items():
    print(f"Test file {quarter}: {path}")

Project dir: /srv/notebooks/[6.0] ML Pricers/Linear_Regression/jewelry/Ridge_Regression
Output dir:  /srv/notebooks/[6.0] ML Pricers/Linear_Regression/jewelry/Ridge_Regression/outputs
Train file:  /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_remainder_training.csv
Test file 2025Q1: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv
Test file 2025Q2: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv
Test file 2025Q3: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv
Test file 2025Q4: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv


## Imports

In [2]:
import pandas as pd

from jewelry_ridge_pricer import (
    apply_feature_schema,
    build_feature_report,
    evaluate_by_price_bucket,
    evaluate_by_quarter,
    evaluate_price_predictions,
    fit_feature_schema,
    fit_ridge_log_price_model,
    get_top_coefficients,
    load_jewelry_data,
    make_model_frame,
    predict_price_from_log_model,
    ridge_training_history,
    split_train_test,
)

## Load and Split Data

In [3]:
missing_inputs = [path for path in [TRAIN_FILE, *TEST_FILES.values()] if not path.exists()]
if missing_inputs:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(str(path) for path in missing_inputs))

raw = load_jewelry_data(
    train_file=TRAIN_FILE,
    test_dir=JEWELRY_DATA_DIR,
    train_before=TRAIN_BEFORE,
    quarters=QUARTERS,
    verbose=VERBOSE,
)
train_raw, test_raw = split_train_test(
    raw,
    train_rows=TRAIN_ROWS,
    test_rows_per_quarter=TEST_ROWS_PER_QUARTER,
    random_state=RANDOM_STATE,
    verbose=VERBOSE,
)

train_keys = set(train_raw["lot_url"].dropna().astype(str))
test_keys = set(test_raw["lot_url"].dropna().astype(str))
print(f"train_rows={len(train_raw):,}")
print(f"test_rows={len(test_raw):,}")
print(f"lot_url_overlap={len(train_keys & test_keys):,}")
test_raw["source_quarter"].value_counts().sort_index()

[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded jewelry frame: 9,580 rows, 65 columns
[jewelry features] Initial split: train=5,580, explicit test=4,000
[jewelry features] De-duplicated train lot_url: 5,580->5,580
[jewelry features] De-duplicated test lot_url: 4,000->4,000
[jewelry features] Removed test-overlapping lot_url values from train: 5,580->5,580
[jewelry features] Final split: train=5,580, test=4,000
train_rows=5,580
test_rows=4,000
lot_url_overlap=0


source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
Name: count, dtype: int64

## Train and Evaluate Variants

In [4]:
predictions = test_raw[
    [c for c in ["ID", "lot_url", "source_file", "source_quarter", "sale_date", TARGET_COL] if c in test_raw]
].copy()
predictions = predictions.rename(columns={TARGET_COL: "actual_price_usd"})

overall_rows = []
quarter_rows = []
bucket_rows = []
history_frames = []
coef_frames = []

for include_estimates, suffix, label in [
    (False, "no_estimates", "Jewelry Ridge - No Estimates"),
    (True, "with_estimates", "Jewelry Ridge - With Log Estimates"),
]:
    train_frame, feature_cols, categorical_cols, numeric_cols = make_model_frame(
        train_raw,
        target_col=TARGET_COL,
        include_estimates=include_estimates,
        verbose=VERBOSE,
    )
    test_frame, _, _, _ = make_model_frame(
        test_raw,
        target_col=TARGET_COL,
        include_estimates=include_estimates,
        verbose=VERBOSE,
    )

    schema = fit_feature_schema(
        train_frame,
        feature_cols,
        categorical_cols,
        min_category_count=MIN_CATEGORY_COUNT,
        verbose=VERBOSE,
    )
    x_train = apply_feature_schema(train_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE)
    x_test = apply_feature_schema(test_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE)

    model = fit_ridge_log_price_model(
        x_train,
        train_frame["log_target_price_usd"],
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        validation_fraction=VALIDATION_FRACTION,
        random_state=RANDOM_STATE,
        verbose=VERBOSE,
    )
    pred = predict_price_from_log_model(model, x_test)
    predictions[f"ridge_pred_{suffix}_usd"] = pred

    overall_rows.append(evaluate_price_predictions(test_frame["target_price_usd"], pred, label=label))
    quarter_rows.append(evaluate_by_quarter(test_frame, pred, actual_col="target_price_usd", label=label))
    bucket_rows.append(evaluate_by_price_bucket(test_frame, pred, actual_col="target_price_usd", label=label))

    feature_report = build_feature_report(train_frame, feature_cols, categorical_cols, schema=schema)
    feature_report.to_csv(OUTPUT_DIR / f"jewelry_ridge_feature_report_{suffix}.csv", index=False)

    history = ridge_training_history(model)
    if not history.empty:
        history.insert(0, "model", label)
        history_frames.append(history)

    coefs = get_top_coefficients(model, top_n=40)
    coefs.insert(0, "model", label)
    coef_frames.append(coefs)

overall = pd.DataFrame(overall_rows)
quarter = pd.concat(quarter_rows, ignore_index=True)
bucket = pd.concat(bucket_rows, ignore_index=True)
overall

[jewelry features] Building jewelry model frame from 5,580 rows using target_col='ah_price_usd', include_estimates=False
[jewelry features] Filtered invalid/non-positive targets: 5,580->5,580
[jewelry features] Engineered 55 features (29 categorical, 26 numeric).
[jewelry features] Building jewelry model frame from 4,000 rows using target_col='ah_price_usd', include_estimates=False
[jewelry features] Filtered invalid/non-positive targets: 4,000->4,000
[jewelry features] Engineered 55 features (29 categorical, 26 numeric).
[jewelry features] Fitting jewelry feature schema with min_category_count=20
[jewelry features]   categorical auction_house: 7 kept levels from 5 observed
[jewelry features]   categorical location: 11 kept levels from 10 observed
[jewelry features]   categorical product_name_clean: 9 kept levels from 3,796 observed
[jewelry features]   categorical brand_clean: 21 kept levels from 501 observed
[jewelry features]   categorical collection_clean: 2 kept levels from 104 ob

/home/vann/.local/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[jewelry features] Applying jewelry feature schema to 4,000 rows and 55 features
[ridge model] Training Ridge on 5,580 rows and 55 features
[ridge model] Alpha grid: [0.001, 0.0029286445646252374, 0.008576958985908946, 0.025118864315095794, 0.07356422544596414, 0.21544346900318845, 0.630957344480193, 1.8478497974222907, 5.411695265464638, 15.848931924611142, 46.41588833612782, 135.93563908785268, 398.1071705534969, 1165.9144011798312, 3414.5488738336007, 10000.0]
[ridge model] Running alpha diagnostics: train_fit=5,022, validation=558, eval_metric=log_RMSE, log_period=1
[ridge model] [01/16] alpha=0.001 log_RMSE=1.10446 log_MAE=0.75487 RMSE=132607908.26 MAE=5656325.37
[ridge model] [02/16] alpha=0.00292864 log_RMSE=1.10435 log_MAE=0.75483 RMSE=132143872.45 MAE=5636680.41
[ridge model] [03/16] alpha=0.00857696 log_RMSE=1.10470 log_MAE=0.75478 RMSE=135141840.34 MAE=5763595.22
[ridge model] [04/16] alpha=0.0251189 log_RMSE=1.10321 log_MAE=0.75441 RMSE=127339285.81 MAE=5433276.45
[ridge mo

/home/vann/.local/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[jewelry features] Applying jewelry feature schema to 4,000 rows and 59 features
[ridge model] Training Ridge on 5,580 rows and 59 features
[ridge model] Alpha grid: [0.001, 0.0029286445646252374, 0.008576958985908946, 0.025118864315095794, 0.07356422544596414, 0.21544346900318845, 0.630957344480193, 1.8478497974222907, 5.411695265464638, 15.848931924611142, 46.41588833612782, 135.93563908785268, 398.1071705534969, 1165.9144011798312, 3414.5488738336007, 10000.0]
[ridge model] Running alpha diagnostics: train_fit=5,022, validation=558, eval_metric=log_RMSE, log_period=1
[ridge model] [01/16] alpha=0.001 log_RMSE=0.29818 log_MAE=0.21626 RMSE=60692.65 MAE=10484.70
[ridge model] [02/16] alpha=0.00292864 log_RMSE=0.29820 log_MAE=0.21627 RMSE=60751.91 MAE=10488.12
[ridge model] [03/16] alpha=0.00857696 log_RMSE=0.29793 log_MAE=0.21621 RMSE=60617.37 MAE=10484.32
[ridge model] [04/16] alpha=0.0251189 log_RMSE=0.29815 log_MAE=0.21626 RMSE=60772.00 MAE=10490.00
[ridge model] [05/16] alpha=0.073

,model,n,MAE,RMSE,MAPE,R2
0,Jewelry Ridge - No Estimates,4000,66326.035090,1.032917e+06,3.501713,-9.050320
1,Jewelry Ridge - With Log Estimates,4000,30705.496084,1.972137e+05,0.563968,0.633628


## Write Outputs

In [5]:
predictions.to_csv(OUTPUT_DIR / "jewelry_ridge_test_predictions.csv", index=False)
overall.to_csv(OUTPUT_DIR / "jewelry_ridge_overall_metrics.csv", index=False)
quarter.to_csv(OUTPUT_DIR / "jewelry_ridge_quarter_metrics.csv", index=False)
bucket.to_csv(OUTPUT_DIR / "jewelry_ridge_price_bucket_metrics.csv", index=False)
if history_frames:
    pd.concat(history_frames, ignore_index=True).to_csv(OUTPUT_DIR / "jewelry_ridge_training_history.csv", index=False)
pd.concat(coef_frames, ignore_index=True).to_csv(OUTPUT_DIR / "jewelry_ridge_top_coefficients.csv", index=False)

print(f"Wrote outputs to {OUTPUT_DIR}")

Wrote outputs to /srv/notebooks/[6.0] ML Pricers/Linear_Regression/jewelry/Ridge_Regression/outputs
